# DL Final Project -- MLP for YouTube View Count Classification

## Goal

Build a MLP to classify YouTube videos into **4 view count categories**.

## Class Labels

| Label | Class | view_count range |
|-------|-------|------------------|
| 0 | Low | < 10,000 |
| 1 | Medium | 10,000 ~ 100,000 |
| 2 | High | 100,000 ~ 300,000 |
| 3 | Viral | ≥ 300,000 |

## Input Features (same as Regression version)

1. `subscriber_count` — 頻道總訂閱數
2. `video_duration_sec` — 影片長度（秒）
3. `categoryId` — 影片分類 ID（排除 10，one-hot encoding）
4. `published_at` — 發布時間（cyclic encoding: hour_sin/cos, weekday_sin/cos）
5. `tag_count` — tags 數量
6. `description_length` — description 字元長度

## Settings

- Train / Test split: 80% / 20%
- Epochs: 200
- Batch size: 64
- Loss: CrossEntropyLoss
- Optimizer: Adam

## Part 1

Clone the dataset from GitHub and import necessary libraries.

In [1]:
!git clone https://github.com/Anson-ntuim/DL-Final.git

Cloning into 'DL-Final'...
remote: Enumerating objects: 17, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 17 (delta 3), reused 15 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (17/17), 10.93 MiB | 12.12 MiB/s, done.
Resolving deltas: 100% (3/3), done.


## Part 2

Import libraries.

In [2]:
# Model
import os
import torch
import torch.nn as nn
from torch.optim.optimizer import Optimizer

# Dataset
from torch.utils.data import Dataset, DataLoader, random_split

# Pre-processing
import pandas as pd
import numpy as np
import json
import re
from datetime import datetime, timezone
from sklearn.metrics import f1_score

## Part 3

Global variables / hyperparameters.

In [3]:
batch_size = 64
num_epoch  = 100
learning_rate = 0.0001
train_ratio   = 0.8
num_classes   = 4    # 4 個分類
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# 分類邊界定義
# Class 0: view_count < 10,000
# Class 1: 10,000 <= view_count < 100,000
# Class 2: 100,000 <= view_count < 300,000
# Class 3: view_count >= 300,000
CLASS_BOUNDARIES = [10_000, 100_000, 300_000]
CLASS_NAMES = ['Low (<10K)', 'Medium (10K~100K)', 'High (100K~300K)', 'Viral (≥300K)']

Using device: cuda


## Part 4

Data loading and preprocessing.

- Parse ISO 8601 duration → seconds
- Parse `published_at` → hour of day + day of week (cyclic encoding)
- Remove rows where `categoryId == 10`
- Fill missing values (tag_count → 0, description_length → 0)
- StandardScaler normalization on all features
- **Target: 4-class label based on view_count thresholds**
  - Class 0: view_count < 10,000
  - Class 1: 10,000 ≤ view_count < 100,000
  - Class 2: 100,000 ≤ view_count < 300,000
  - Class 3: view_count ≥ 300,000

In [4]:
# ── 工具函式 ────────────────────────────────────────────────

def parse_iso8601_duration(duration_str):
    """ISO 8601 duration (e.g. PT1H2M3S) → seconds (int)"""
    if not isinstance(duration_str, str):
        return 0
    pattern = r'PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?'
    match = re.match(pattern, duration_str)
    if not match:
        return 0
    h = int(match.group(1) or 0)
    m = int(match.group(2) or 0)
    s = int(match.group(3) or 0)
    return h * 3600 + m * 60 + s


def parse_published_at(dt_str):
    """ISO 8601 datetime → (hour_of_day, day_of_week)"""
    try:
        dt = datetime.fromisoformat(dt_str.replace('Z', '+00:00'))
        return dt.hour, dt.weekday()   # 0-23, 0-6
    except Exception:
        return 0, 0


def view_count_to_label(view_count):
    """view_count → class label (0, 1, 2, 3)"""
    if view_count < 10_000:
        return 0
    elif view_count < 100_000:
        return 1
    elif view_count < 300_000:
        return 2
    else:
        return 3


class StandardScaler:
    def fit(self, X):
        self.mean_ = X.mean(axis=0)
        self.std_  = X.std(axis=0) + 1e-8
        return self
    def transform(self, X):
        return (X - self.mean_) / self.std_


# ── 讀取資料 ────────────────────────────────────────────────
data_dir = 'DL-Final/data'
files = os.listdir(data_dir)

csv_files  = [f for f in files if f.endswith('.csv')]
json_files = [f for f in files if f.endswith('.json')]

if csv_files:
    df = pd.read_csv(os.path.join(data_dir, csv_files[0]))
elif json_files:
    df = pd.read_json(os.path.join(data_dir, json_files[0]))
else:
    raise FileNotFoundError('找不到資料檔')

print(f'Loaded data shape={df.shape}')

Loaded data shape=(34383, 27)


/tmp/ipykernel_2296/1780031251.py:55: DtypeWarning: Columns (18) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(os.path.join(data_dir, csv_files[0]))


In [5]:
# ── 特徵工程 ────────────────────────────────────────────────

# 1. 移除 category_id == 10
if 'category_id' in df.columns:
    df = df[df['category_id'] != 10].copy()

# 2. 處理類別型特徵：One-hot Encoding for category_id
df['category_id'] = df['category_id'].astype(str)
df_encoded = pd.get_dummies(df, columns=['category_id'], prefix='cat')

# 3. 數值特徵轉換
if 'duration_iso8601' in df.columns:
    df_encoded['video_duration_sec'] = df['duration_iso8601'].apply(parse_iso8601_duration)

if 'published_at' in df.columns:
    df_encoded[['pub_hour', 'pub_weekday']] = df['published_at'].apply(
        lambda x: pd.Series(parse_published_at(str(x)))
    )

    # 週期性編碼 (Cyclic Encoding) for pub_hour and pub_weekday
    df_encoded['pub_hour_sin']     = np.sin(2 * np.pi * df_encoded['pub_hour']    / 24.0)
    df_encoded['pub_hour_cos']     = np.cos(2 * np.pi * df_encoded['pub_hour']    / 24.0)
    df_encoded['pub_weekday_sin']  = np.sin(2 * np.pi * df_encoded['pub_weekday'] / 7.0)
    df_encoded['pub_weekday_cos']  = np.cos(2 * np.pi * df_encoded['pub_weekday'] / 7.0)

# subscriber_count / duration / description 壓縮至 log scale
df_encoded['subscriber_count']   = np.log1p(df_encoded['subscriber_count'])
df_encoded['video_duration_sec'] = np.log1p(df_encoded['video_duration_sec'])
df_encoded['description_length'] = np.log1p(df_encoded['description_length'])
df_encoded['channel_view_count'] = np.log1p(df_encoded['channel_view_count'])
df_encoded['channel_video_count'] = np.log1p(df_encoded['channel_video_count'])

df['days_since_published'] = df['published_at'].apply(
    lambda x: (datetime.now(timezone.utc) -
               datetime.fromisoformat(str(x).replace('Z','+00:00'))).days
    if 'T' in str(x) else 0
)
df_encoded['days_since_published'] = np.log1p(df['days_since_published'])

# 4. 建立特徵清單 (包含 One-hot 後的欄位)
CAT_COLS = [col for col in df_encoded.columns if col.startswith('cat_')]
NUM_COLS = ['subscriber_count', 'video_duration_sec', 'tags_count', 'description_length',
            'pub_hour_sin', 'pub_hour_cos', 'pub_weekday_sin', 'pub_weekday_cos','days_since_published',
            'channel_view_count','channel_video_count']
FEATURE_COLS = CAT_COLS + NUM_COLS

X = df_encoded[FEATURE_COLS].values.astype(np.float32)
input_dim = X.shape[1]

# ── 建立分類標籤 ────────────────────────────────────────────
view_col = 'view_count' if 'view_count' in df.columns else 'viewCount'
y = df[view_col].values.astype(np.int64)
y = np.array([view_count_to_label(v) for v in y], dtype=np.int64)

print(f'Feature matrix shape: {X.shape}')
print(f'Label distribution:')
for i, name in enumerate(CLASS_NAMES):
    count = (y == i).sum()
    print(f'  Class {i} [{name}]: {count} samples ({count/len(y)*100:.1f}%)')

display(df_encoded[FEATURE_COLS].head(5))

Feature matrix shape: (34383, 25)
Label distribution:
  Class 0 [Low (<10K)]: 28188 samples (82.0%)
  Class 1 [Medium (10K~100K)]: 4936 samples (14.4%)
  Class 2 [High (100K~300K)]: 893 samples (2.6%)
  Class 3 [Viral (≥300K)]: 366 samples (1.1%)


,cat_1,cat_15,cat_17,cat_19,cat_2,cat_20,cat_22,cat_23,cat_24,cat_25,...,video_duration_sec,tags_count,description_length,pub_hour_sin,pub_hour_cos,pub_weekday_sin,pub_weekday_cos,days_since_published,channel_view_count,channel_video_count
0,False,False,False,True,False,False,False,False,False,False,...,5.789960,0,3.583519,0.866025,0.500000,0.000000,1.000000,6.278521,16.097106,8.523374
1,False,False,False,False,False,True,False,False,False,False,...,9.770870,0,5.375278,-0.965926,-0.258819,-0.781831,0.623490,4.060443,17.034060,6.061457
2,False,False,False,False,False,False,True,False,False,False,...,3.465736,0,5.192957,0.707107,0.707107,-0.433884,-0.900969,5.572154,13.447042,4.127134
3,False,False,False,False,False,False,True,False,False,False,...,2.397895,52,5.541264,-0.258819,-0.965926,0.000000,1.000000,4.844187,16.226385,8.687273
4,False,False,False,False,False,False,False,False,True,False,...,7.534228,0,5.605802,-0.707107,0.707107,0.781831,0.623490,7.278629,11.694413,5.093750


## Part 5

Create Dataset and DataLoader with train/test split (80/20).

In [6]:
class YouTubeDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)   # long 型別供 CrossEntropyLoss 使用

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

n_total = len(X)
n_train = int(n_total * train_ratio)
indices = np.random.permutation(n_total)
train_idx = indices[:n_train]
test_idx  = indices[n_train:]

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

scaler  = StandardScaler().fit(X_train)
X_train = scaler.transform(X_train)
X_test  = scaler.transform(X_test)

train_dataset = YouTubeDataset(X_train, y_train)
test_dataset  = YouTubeDataset(X_test,  y_test)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False)

print(f'Train samples: {len(train_idx)} | Test samples: {len(test_idx)}')

Train samples: 27506 | Test samples: 6877


## Part 6

Activation function — ReLU (custom implementation).

In [7]:
class myActivation(nn.Module):

    def __init__(self):
        super().__init__()
        self.relu = nn.ReLU()

    def forward(self, x):
        # ReLU: max(0, x)
        return self.relu(x)

## Part 7

MLP model architecture.

- Input: same features as regression version
- Hidden layers: 1024 → 512 → 256 → 64
- **Output: 4 (classification logits, one per class)**
- Activation: ReLU
- BatchNorm for stable training

In [8]:
class myMLP(nn.Module):
    def __init__(self, input_dim, num_classes=4,dropoutRate=0.1):
        super(myMLP, self).__init__()
        self.mlp = nn.Sequential(

          #  'Model_C': [1024, 512, 256, 128, 64],  lr=0.0001  wd=1e-03  dr=0.1
            nn.Linear(input_dim, 1024),
            nn.BatchNorm1d(1024),
            myActivation(),
            nn.Dropout(dropoutRate),

            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            myActivation(),
            nn.Dropout(dropoutRate),

            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            myActivation(),
            nn.Dropout(dropoutRate),

            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            myActivation(),
            nn.Dropout(dropoutRate),

            nn.Linear(128, 64),
            myActivation(),

            nn.Linear(64, num_classes)   # 輸出 4 個 logits (不加 Softmax，由 CrossEntropyLoss 處理)
        )

    def forward(self, x):
        return self.mlp(x)

model = myMLP(input_dim, num_classes=num_classes).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {total_params:,}')

Total parameters: 728,004


## Part 8

Loss function — **Cross Entropy Loss** for multi-class classification.

In [9]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.arange(num_classes)
weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weights = torch.tensor(weights, dtype=torch.float32).to(device)

class myLoss(nn.Module):
    def __init__(self, weight=None):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(weight=weight)

    def forward(self, pred, target):
        return self.ce(pred, target)

criterion = myLoss(weight=class_weights)

NameError: name 'NUM_CLASSES' is not defined

## Part 9

Optimizer — Adam with weight decay.

In [ ]:
import torch.optim as optim

optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-3)

## Part 10

Model training.

In [ ]:
model.train()

# 初始化學習率排程器
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=15)

for epoch in range(num_epoch):

    losses = []

    for batch_num, (x, y_batch) in enumerate(train_loader):
        optimizer.zero_grad()

        x = x.to(device).float()
        y_batch = y_batch.to(device).long()   # classification 使用 long

        output = model(x)         # (batch, 4) logits
        loss = criterion(output, y_batch)
        loss.backward()

        # 梯度裁剪防止極端數值導致訓練不穩
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        losses.append(loss.item())
        optimizer.step()

    avg_loss = sum(losses) / len(losses)
    print(f'Epoch {epoch} | Avg Loss {avg_loss:.4f}')
    scheduler.step(avg_loss)

torch.save(model.state_dict(), 'final_mlp_cls_model_state_dict.pt')
print('\nModel state dict saved as final_mlp_cls_model_state_dict.pt')

## Part 11

Model evaluation on test set.

Metrics:
- **Accuracy** — overall
- **Per-class Accuracy**
- **Confusion Matrix**

In [ ]:
model.eval()

all_preds  = []
all_labels = []

with torch.no_grad():
    for x, y_batch in test_loader:
        x = x.to(device).float()
        y_batch = y_batch.to(device).long()

        logits = model(x)                 # (batch, 4)
        preds  = torch.argmax(logits, dim=1)      # predicted class

        all_preds.append(preds.cpu())
        all_labels.append(y_batch.cpu())

all_preds  = torch.cat(all_preds,  dim=0)
all_labels = torch.cat(all_labels, dim=0)

# Overall Accuracy
overall_acc = (all_preds == all_labels).float().mean().item()

print('=' * 50)
print(f'Test Overall Accuracy: {overall_acc * 100:.2f}%')
print('=' * 50)

# Per-class Accuracy
print('\nPer-class Accuracy:')
for i, name in enumerate(CLASS_NAMES):
    mask = (all_labels == i)
    if mask.sum() == 0:
        print(f'  Class {i} [{name}]: N/A (no samples)')
    else:
        class_acc = (all_preds[mask] == all_labels[mask]).float().mean().item()
        print(f'  Class {i} [{name}]: {class_acc * 100:.2f}% ({mask.sum().item()} samples)')


# Calculate Macro F1-score
macro_f1 = f1_score(all_labels.numpy(), all_preds.numpy(), average='macro')
print(f'\nMacro F1-score: {macro_f1:.4f}')

## Part 12

Visualize results — Confusion Matrix and class distribution.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

preds_np  = all_preds.numpy()
labels_np = all_labels.numpy()

# ── Confusion Matrix ────────────────────────────────────────
conf_matrix = np.zeros((num_classes, num_classes), dtype=int)
for t, p in zip(labels_np, preds_np):
    conf_matrix[t][p] += 1

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# 左圖：Confusion Matrix
im = axes[0].imshow(conf_matrix, cmap='Blues')
axes[0].set_xticks(range(num_classes))
axes[0].set_yticks(range(num_classes))
axes[0].set_xticklabels([f'Pred {i}\n{n}' for i, n in enumerate(CLASS_NAMES)], fontsize=8)
axes[0].set_yticklabels([f'True {i}\n{n}' for i, n in enumerate(CLASS_NAMES)], fontsize=8)
for i in range(num_classes):
    for j in range(num_classes):
        axes[0].text(j, i, str(conf_matrix[i][j]),
                     ha='center', va='center',
                     color='white' if conf_matrix[i][j] > conf_matrix.max() / 2 else 'black')
axes[0].set_xlabel('Predicted Label')
axes[0].set_ylabel('True Label')
axes[0].set_title(f'Confusion Matrix\n(Accuracy: {overall_acc*100:.2f}%)')
plt.colorbar(im, ax=axes[0])

# 右圖：Class distribution (真實 vs 預測)
x_pos = np.arange(num_classes)
true_counts = [(labels_np == i).sum() for i in range(num_classes)]
pred_counts = [(preds_np  == i).sum() for i in range(num_classes)]

width = 0.35
axes[1].bar(x_pos - width/2, true_counts, width, label='Ground Truth', color='steelblue', alpha=0.8)
axes[1].bar(x_pos + width/2, pred_counts, width, label='Predicted',    color='coral',     alpha=0.8)
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels([f'Class {i}\n{n}' for i, n in enumerate(CLASS_NAMES)], fontsize=8)
axes[1].set_ylabel('Count')
axes[1].set_title('Class Distribution: Ground Truth vs Predicted')
axes[1].legend()

plt.suptitle('MLP YouTube View Count Classification', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('classification_analysis.png', dpi=120, bbox_inches='tight')
plt.show()
print('Plot saved as classification_analysis.png')